# 14. Psychoacoustics and codecs

Telephone speech, MP3, AAC and Opus all throw information away, and they get away with it
because the ear cannot hear everything that is in the signal: quiet sounds near loud ones are
masked, very quiet sounds fall below the threshold of hearing, and loudness is perceived on a
roughly logarithmic scale. This notebook puts numbers on those effects and implements G.711
μ-law companding from the standard.

**Learning outcomes.** After this notebook you can

- distinguish dB SPL from dBFS and explain why a calibration is always an assumption;
- convert between Hz and the mel (HTK and Slaney), Bark and ERB scales, and say which mel
  the library uses where;
- evaluate Terhardt's threshold in quiet and a spreading function for simultaneous masking;
- implement μ-law companding and show why it keeps the SNR roughly constant over 40 dB.

*Note.* The perceptual-codec, Bark, ERB and threshold-of-hearing functions of the Rust crate
(`audio_samples::codecs`) are not exposed in the Python bindings, so this notebook computes
them with NumPy from the same published formulas (the Rust chapter checks the Rust functions
against the same numbers).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import audio_samples as aus

rng = np.random.default_rng(0)
F64 = np.dtype("float64")  # the generators take a numpy dtype instance

## Concept

**dBFS is not dB SPL.** Sound pressure level is an absolute physical quantity,

$$L_p = 20 \log_{10}(p / p_0)\ \text{dB SPL}, \qquad p_0 = 20\ \mu\text{Pa}$$

where $p$ is the RMS sound pressure in pascals; 1 Pa is 93.98 dB SPL. Digital audio only knows
**dBFS**, decibels relative to digital full scale ($20\log_{10}$ of an amplitude, full
scale = 1). The link between the two is the whole playback chain (converter, amplifier, volume
knob, transducer, distance), so a file has no absolute level; a psychoacoustic model that
compares a signal with the threshold of hearing must *assume* a calibration.

**Perceptual frequency scales.** The ear's frequency resolution is roughly constant in Hz below
about 500 Hz and roughly proportional to frequency above. With $f$ in Hz:

- **Mel, HTK form:** $m = 2595 \log_{10}(1 + f/700)$ mel (1000 Hz ≈ 1000 mel). This is what
  `aus.utils.hz_to_mel` computes. **Slaney form** (librosa's default, and what the library's
  *mel spectrogram* uses): linear, $m = 3f/200$, below 1 kHz and logarithmic above.
- **Bark (Traunmüller 1990):** $z = 26.81 f / (1960 + f) - 0.53$ Bark. One Bark is about one
  critical band.
- **ERB (Glasberg and Moore 1990):** the equivalent rectangular *bandwidth* of the auditory
  filter at $f$ is $\mathrm{ERB}(f) = 24.7 (4.37 f / 1000 + 1)$ Hz; the perceptual *coordinate*
  is the ERB-number $E(f) = 21.4 \log_{10}(1 + 0.00437 f)$ Cam.

**Absolute threshold of hearing** (Terhardt 1979), in dB SPL with $k = f/1000$:

$$T_q(f) = 3.64\, k^{-0.8} - 6.5\, e^{-0.6 (k - 3.3)^2} + 10^{-3} k^4$$

High at low frequencies, a minimum of about $-5$ dB SPL near 3.3 kHz (the ear-canal
resonance), rising again above 10 kHz.

**Simultaneous masking.** A loud component raises the threshold for components nearby in
frequency. On the Bark scale the masked threshold falls off steeply *below* the masker and
more gently *above* it: masking spreads upwards. The spreading function of Schroeder, Atal and
Hall (1979) is

$$S(\Delta z) = 15.81 + 7.5(\Delta z + 0.474) - 17.5\sqrt{1 + (\Delta z + 0.474)^2}\ \text{dB}$$

where $\Delta z$ is the maskee's Bark position minus the masker's. Far from the masker its
slopes approach $-10$ dB/Bark above and $+25$ dB/Bark below. A perceptual coder may add
quantisation noise up to the masked threshold without it being audible.

**μ-law companding (ITU-T G.711).** Uniform quantisation gives quiet talkers a poor SNR
(6.02 dB per bit, minus every dB the signal sits below full scale; notebook 02). Companding
quantises the *logarithm* of the amplitude instead. The continuous μ-law curve is

$$F(x) = \mathrm{sgn}(x)\, \frac{\ln(1 + \mu |x|)}{\ln(1 + \mu)}, \qquad \mu = 255$$

for $x \in [-1, 1]$. G.711 implements a segmented approximation: a **14-bit** linear sample
becomes an **8-bit** code (sign bit, 3-bit segment, 4-bit mantissa). The magnitude plus a bias
of 33 falls in one of 8 segments, each twice as wide as the last, with 16 equal steps per
segment: step sizes 2, 4, ..., 256 (14-bit units).

*The library's `mu_compress` currently deviates from this formula (see the CHANGELOG), so this
notebook implements the curve itself and uses only the library's `mu_expand`, which inverts
the standard curve.*

## With the library

**Mel, two ways.** `aus.utils.hz_to_mel` is the HTK formula and agrees with
`librosa.hz_to_mel(..., htk=True)`; `mel_to_hz` inverts it.

In [ ]:
u = aus.utils
assert abs(u.hz_to_mel(1_000.0) - 1_000.0) < 0.02  # 999.99 mel
for f in (100.0, 1_000.0, 4_000.0):
    assert abs(u.hz_to_mel(f) - librosa.hz_to_mel(f, htk=True)) < 1e-9
    assert abs(u.mel_to_hz(u.hz_to_mel(f)) - f) < 1e-9
# mel_frequencies is HTK-spaced too.
assert np.allclose(u.mel_frequencies(5, 0.0, 8_000.0),
                   librosa.mel_frequencies(5, fmin=0.0, fmax=8_000.0, htk=True), atol=1e-9)
# Slaney (librosa's default) is a different scale: 1000 Hz is 15 "Slaney mel".
assert abs(librosa.hz_to_mel(1_000.0) - 15.0) < 1e-9

The **mel spectrogram**, however, uses Slaney filters. Put a pure tone on FFT bin $k$, find the
mel band with the most energy, and compare with the band whose Slaney (and whose HTK) triangle
peaks highest at that bin. Below 1 kHz the two scales disagree on dozens of bins; the library
always sides with Slaney.

In [ ]:
sr, n_fft, n_mels = 16_000, 512, 40
spec_params = aus.SpectrogramParams(
    aus.StftParams(n_fft=n_fft, hop_size=256, window=aus.WindowType.hanning, centre=True), float(sr))
mel_params = aus.MelParams(n_mels=n_mels, f_min=0.0, f_max=8_000.0)
fb_slaney = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels, fmin=0, fmax=8_000, htk=False, norm=None)
fb_htk = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels, fmin=0, fmax=8_000, htk=True, norm=None)

agree_slaney = agree_htk = disputed = 0
for k in range(2, 256, 2):  # tones exactly on FFT bins
    tone = aus.sine_wave(k * sr / n_fft, 0.25, sample_rate=sr, dtype=F64)
    mel = np.asarray(tone.mel_spectrogram(spec_params, mel_params, None))
    band = np.argmax(mel[:, mel.shape[1] // 2])
    s, h = np.argmax(fb_slaney[:, k]), np.argmax(fb_htk[:, k])
    if s != h:
        disputed += 1
        agree_slaney += band == s
        agree_htk += band == h
print(f"{disputed} bins where the scales disagree: Slaney wins {agree_slaney}, HTK wins {agree_htk}")
assert disputed > 50 and agree_slaney == disputed and agree_htk == 0

**μ-law expansion.** Compress with the standard curve (NumPy), expand with the library: the
composition is the identity.

In [ ]:
mu = 255.0


def mu_law(x, mu=255.0):
    return np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)


xs = np.array([-0.9, -0.3, -0.01, 0.0, 0.02, 0.5, 1.0])
back = aus.AudioSamples.new_mono(mu_law(xs), 8_000).mu_expand(mu).to_numpy()
assert np.allclose(back, xs, rtol=0, atol=1e-12)  # identity to rounding

## Under the hood

**The scales.** 1 kHz is 8.53 Bark and 4 kHz is 17.46 Bark; the ERB bandwidth is 132.6 Hz at
1 kHz and 456.5 Hz at 4 kHz; the ERB-number at 1 kHz is 15.62 Cam.

In [ ]:
def hz_to_bark(f):
    return 26.81 * f / (1960 + f) - 0.53


def bark_to_hz(z):
    return 1960 * (z + 0.53) / (26.28 - z)


erb_bandwidth = lambda f: 24.7 * (4.37 * f / 1000 + 1)
erb_number = lambda f: 21.4 * np.log10(1 + 0.00437 * f)

assert abs(hz_to_bark(1_000) - 8.527) < 1e-3 and abs(hz_to_bark(4_000) - 17.46) < 0.01
assert abs(bark_to_hz(hz_to_bark(1_000.0)) - 1_000) < 1e-9
assert abs(erb_bandwidth(1_000) - 132.639) < 1e-3 and abs(erb_bandwidth(4_000) - 456.456) < 1e-3
assert abs(erb_number(1_000) - 15.62) < 0.01

f = np.linspace(20, 8_000, 800)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(f, librosa.hz_to_mel(f, htk=True) / librosa.hz_to_mel(8_000, htk=True), label="mel (HTK)")
ax.plot(f, librosa.hz_to_mel(f) / librosa.hz_to_mel(8_000), label="mel (Slaney)")
ax.plot(f, hz_to_bark(f) / hz_to_bark(8_000), label="Bark")
ax.plot(f, erb_number(f) / erb_number(8_000), label="ERB-number")
ax.set(xlabel="Frequency (Hz)", ylabel="Scale value / value at 8 kHz",
       title="Perceptual scales, normalised at 8 kHz")
ax.legend(fontsize=8)
plt.show()

**Threshold of hearing and spreading.** Terhardt's curve is 22.95 dB SPL at 100 Hz, 3.37 dB SPL
at 1 kHz, and has its minimum of $-4.98$ dB SPL near 3.32 kHz. The Schroeder spreading
function's slopes far from the masker are $-10$ dB/Bark above and $+25$ dB/Bark below.

In [ ]:
def ath(f):
    k = np.asarray(f, dtype=float) / 1000
    return 3.64 * k**-0.8 - 6.5 * np.exp(-0.6 * (k - 3.3) ** 2) + 1e-3 * k**4


assert abs(ath(100) - 22.95) < 0.01 and abs(ath(1_000) - 3.37) < 0.01 and abs(ath(3_300) + 4.98) < 0.01
grid = np.arange(20, 16_001, 10)
i = np.argmin(ath(grid))
assert abs(grid[i] - 3_320) <= 20 and abs(ath(grid)[i] + 4.98) < 0.01


def spread(dz):
    return 15.81 + 7.5 * (dz + 0.474) - 17.5 * np.sqrt(1 + (dz + 0.474) ** 2)


upper = (spread(10.0) - spread(8.0)) / 2  # dB per Bark, far above the masker
lower = (spread(-8.0) - spread(-10.0)) / 2  # dB per Bark, far below (rising towards it)
assert abs(upper + 10) < 0.3 and abs(lower - 25) < 0.3

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
fq = np.geomspace(20, 16_000, 400)
axes[0].semilogx(fq, ath(fq))
axes[0].set(xlabel="Frequency (Hz)", ylabel="Threshold (dB SPL)", ylim=(-10, 80),
            title="Threshold in quiet (Terhardt)")
dz = np.linspace(-4, 8, 400)
axes[1].plot(dz, spread(dz))
axes[1].set(xlabel="Maskee - masker (Bark)", ylabel="S (dB)", title="Spreading (Schroeder)")
fig.tight_layout()
plt.show()

**G.711 μ-law, from the standard.** Encode a 14-bit sample to an 8-bit code and back. The error
never exceeds half a step, the code tracks $127\,F(x)$ to within 2 codes, and the SNR of a sine
stays between 34 and 40.5 dB over 40 dB of level, while 8-bit linear PCM starts at
$6.02 \cdot 8 + 1.76 = 49.9$ dB and loses about 40 dB.

In [ ]:
def ulaw_encode(x):
    mag, sign = (-x, 0x80) if x < 0 else (x, 0x00)
    biased = min(min(mag, 8_159) + 33, 8_191)  # clip, bias, saturate
    seg = next(s for s in range(8) if biased < (64 << s))  # segment 0..7
    mant = (biased >> (seg + 1)) & 0x0F  # 4-bit mantissa
    return ~(sign | (seg << 4) | mant) & 0xFF  # G.711 transmits the code inverted


def ulaw_decode(code):
    c = ~code & 0xFF
    seg, mant = (c >> 4) & 7, c & 0x0F
    mag = (((mant << 1) + 33) << seg) - 33  # middle of the step
    return -mag if c & 0x80 else mag


assert ulaw_encode(0) == 0xFF  # positive zero
assert ulaw_decode(ulaw_encode(8_159)) == 8_031  # largest magnitude
xs14 = np.arange(-8_159, 8_160)
dec = np.array([ulaw_decode(ulaw_encode(int(v))) for v in xs14])
seg = np.array([next(s for s in range(8) if min(abs(int(v)) + 33, 8_191) < (64 << s)) for v in xs14])
assert np.all(np.abs(xs14 - dec) <= 2.0**seg)  # at most half a step, 2^seg

pos = np.arange(1, 8_160)
idx = np.array([~ulaw_encode(int(v)) & 0x7F for v in pos])  # segment/mantissa index 0..127
assert np.max(np.abs(idx - 127 * mu_law(pos / 8_159))) < 2.0


def snr_db(level_db, q):
    x = 10 ** (level_db / 20) * np.sin(2 * np.pi * 0.1234567 * np.arange(8_000))
    return 10 * np.log10(np.sum(x**2) / np.sum((x - q(x)) ** 2))


ulaw_q = lambda x: np.array([ulaw_decode(ulaw_encode(int(round(v * 8_159)))) for v in x]) / 8_159
lin8_q = lambda x: np.round(x * 127) / 127
levels = [0.0, -10.0, -20.0, -30.0, -40.0]
su = np.array([snr_db(l, ulaw_q) for l in levels])
sl = np.array([snr_db(l, lin8_q) for l in levels])
print("mu-law SNR (dB):", np.round(su, 1))
print("8-bit   SNR (dB):", np.round(sl, 1))
assert np.all((su >= 34.0) & (su < 40.5))
assert abs(sl[0] - 49.9) < 0.5 and abs(sl[0] - sl[4] - 40.0) < 1.5
assert su[4] > sl[4] + 20

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(levels, su, "o-", label="G.711 mu-law (8 bits)")
ax.plot(levels, sl, "s-", label="linear PCM (8 bits)")
ax.set(xlabel="Sine peak level (dBFS)", ylabel="SNR (dB)", title="Companding keeps the SNR flat")
ax.legend(fontsize=8)
plt.show()

## Ask the library to explain

There is no explainer for psychoacoustics or codecs yet. The closest is the quantisation topic,
which gives the uniform-quantiser SNR that μ-law improves on for quiet signals:
$6.02 \cdot 8 + 1.76 = 49.9$ dB for a full-scale sine at 8 bits.

In [ ]:
doc = aus.explain.quantisation(8)
doc

In [ ]:
assert abs(doc.quantity("Rule of thumb") - 49.92) < 0.01
assert abs(doc.quantity("SQNR (full-scale sine)") - 49.92) < 0.1
assert abs(doc.quantity("SQNR (full-scale sine)") - sl[0]) < 0.5  # our 8-bit measurement

## Exercises

**1. Where is 4 kHz?** Predict roughly how many Bark lie between 1 kHz and 4 kHz, then compute
it. Put the difference in `answer`.

In [ ]:
answer = None  # TODO

In [ ]:
# Solution
answer = hz_to_bark(4_000) - hz_to_bark(1_000)
# About 9 critical bands for two octaves, while 0 to 1 kHz is 8.5 Bark.
assert abs(answer - 8.93) < 0.01

**2. Threshold at 50 Hz.** Is a 50 Hz tone at 30 dB SPL audible in quiet according to
Terhardt's formula? Put the threshold (dB SPL) in `answer`.

In [ ]:
answer = None  # TODO

In [ ]:
# Solution
answer = float(ath(50))
assert abs(answer - 40.0) < 0.1
assert 30 < answer  # about 10 dB below threshold: inaudible

**3. Smallest and largest μ-law steps.** What is the ratio between the largest and the smallest
G.711 step, in dB? Why does this keep the SNR roughly constant? Put the ratio in dB in
`answer`.

In [ ]:
answer = None  # TODO

In [ ]:
# Solution
steps = [2 << s for s in range(8)]
assert (steps[0], steps[-1]) == (2, 256)
# Check against the codec: adjacent decoded levels differ by the step size.
levels_pos = sorted({ulaw_decode(c) for c in range(256) if ulaw_decode(c) >= 0})
gaps = np.diff(levels_pos)
assert gaps.max() == 256 and gaps[1:].min() == 2  # (the first gap straddles zero)
answer = 20 * np.log10(256 / 2)
# The step grows in proportion to the amplitude, so the noise grows with the signal and the
# SNR stays roughly constant (until the signal falls into segment 0).
assert abs(answer - 42.14) < 0.01

**4. Calibration.** A recording's speech peaks at $-12$ dBFS. The playback chain is calibrated
so that a full-scale sine (RMS $-3.01$ dBFS) plays at 94 dB SPL. What SPL do the speech peaks
correspond to (as a sine of that peak)? Put it in `answer`.

In [ ]:
answer = None  # TODO

In [ ]:
# Solution
calibration_spl = 94.0  # full-scale sine
sine12 = aus.sine_wave(1_000.0, 1.0, sample_rate=16_000, amplitude=10 ** (-12 / 20), dtype=F64)
full = aus.sine_wave(1_000.0, 1.0, sample_rate=16_000, amplitude=1.0, dtype=F64)
rms_offset = 20 * np.log10(sine12.rms() / full.rms())  # same shape: RMS offset = peak offset
answer = calibration_spl + rms_offset
assert abs(20 * np.log10(sine12.rms()) + 15.01) < 0.01  # its RMS is -15.01 dBFS
assert abs(answer - 82.0) < 1e-9
# And 1 Pa is 93.98 dB SPL.
assert abs(20 * np.log10(1 / 20e-6) - 93.98) < 0.01

## Common pitfalls

- **dBFS conventions.** Some tools quote the RMS of a full-scale sine as 0 dBFS (AES17); this
  library uses $20\log_{10}(\text{RMS})$, giving $-3.01$ dBFS.
- **"Mel" is ambiguous.** `aus.utils.hz_to_mel` / `mel_to_hz` / `mel_frequencies` are HTK;
  the mel spectrogram uses Slaney filters; librosa defaults to Slaney. Pass `htk=True` to
  librosa when comparing with `aus.utils`.
- **ERB bandwidth vs ERB-number.** A bandwidth in Hz is not a perceptual coordinate:
  uniformly spaced ERB *bandwidths* are uniformly spaced in Hz.
- **Masking spreads upwards in frequency.** Check the slopes of any masking model you use.
- **`mu_compress` deviates from the μ-law formula** at present; implement the curve yourself
  (as here) until it is fixed. `mu_expand` is correct.
- **The library's perceptual and "Opus" codecs** (Rust only) are teaching codecs, not
  bitstream compatible with MP3 or Opus.

## See also

- Rust guide chapter: <https://docs.rs/audio_samples/latest/audio_samples/guide/c14_psychoacoustics_codecs/index.html>
  (Bark, ERB, threshold of hearing, masking and the perceptual codec in Rust), and the lab
  `examples/tutorial_14_psychoacoustics_codecs.rs`.
- Python API: `aus.utils.hz_to_mel`, `aus.utils.mel_to_hz`, `aus.utils.mel_frequencies`,
  `AudioSamples.mel_spectrogram`, `AudioSamples.mu_expand`, `aus.explain.quantisation`.
- Previous: [13. Onsets and tempo](13_onsets_tempo.ipynb); quantisation basics:
  [02. Quantisation and dither](02_quantisation_dither.ipynb); the mel scale in practice:
  [10. Cepstrum, mel and MFCC](10_cepstrum_mel_mfcc.ipynb).